# ⚡ Google Drive GPU Refiner - Remote Worker Node (Helper)

This lightweight helper notebook provides external GPU compute (NVIDIA Tesla T4 / V100 / A100 / L4) to accelerate video refining for the **Main Google Drive Downloader & GPU Refiner**.

---

### 🌟 How it Works:
1. **Shared Drive Integration**: Mounts the shared `downloader` Google Drive folder.
2. **Zero Configuration**: Automatically discovers the Main coordinator via `data/cluster/master.json` on Google Drive.
3. **Hardware NVENC Acceleration**: Receives 3–5 minute video chunks, transcodes them at maximum GPU speed using NVENC hardware acceleration, and saves the finished segments directly back into Google Drive.
4. **Fault Tolerant**: If this Colab runtime stops or is preempted, the Main coordinator automatically reclaims the chunk and reassigns it to another worker without losing any completed work.


In [ ]:
#@title 🛠️ Step 1: Environment Setup & Google Drive Mount
#@markdown Mounts Google Drive, verifies NVIDIA GPU hardware acceleration, and prepares workspace.

import os
import sys
import shutil
import subprocess
import json
import base64
import zlib

print("=" * 72)
print("🚀 [1/3] Checking Hardware Acceleration (NVIDIA GPU)...")
print("=" * 72)
has_gpu = False
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ Active GPU Detected: {gpu_info}")
    has_gpu = True
except Exception:
    print("⚠️ No NVIDIA GPU detected! Helper will run in CPU fallback mode (slower).")
    print("👉 Recommendation: Click Runtime > Change runtime type > Select T4 GPU.")

print("\n" + "=" * 72)
print("📁 [2/3] Mounting Google Drive...")
print("=" * 72)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n" + "=" * 72)
print("📦 [3/3] Deploying Helper Modules...")
print("=" * 72)
workspace_dir = "/content/colab-gpu-helper"
os.makedirs(workspace_dir, exist_ok=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# Decompress and write modular files
bundle_b64 = "eJztfQtb20iW6F+pJrcHu9sYyKNnmllmLwF3wjSBLCbJzgdcfbItgxpZ8ugB8bD893seVVJVqSTbpGf33v3W/XWwpXqeOnXedepxY5zE0/CmP19s7ImNcDZP0lwk2VUsv6ZB+bUowkn5Iw9n1ZvstsjDqHp3mwb+JIxvruJpmsxEvpjDDyHfns3zMIn9qCdOwizviaNwDP8exAtZeu7nt1E4UsU/ws+r+Cp+IY6CPBjnYhDfh2kSz4I43xPvkuQmCsRhEvkjcZ+Jk2Tsw0COh97h2cnBW7EPc+lji/3gK/SWda42tmHGOVS+2ugKP56IOMnrhQ73rvCz0cWuJ8EU4JAl0X3gTZKHOEr8SZB6kzDtdMXWX0SWp3tXsYDP1Qb9R9+PFrE/C2E40UJEMKw8yNRwj9LwPhBVUwKagqkl6aInsmKO80aAZbfwZVzkWY9b7KsHW7mf3gR5tjVabIUTHluBUO3RjLJbPw0mYppE0HYmHm5D6POm8FMfJh5gy1N4knGjUFScDj4PzsVDGuYAGDkYGHWemANOkyTv1ycaTgmGCuYSFPgZ+VmAYNKWwR9l+LejfsNbAFPQ8Twck+d1u1X1NMiLNC6r/paEcUc12YMhaGuR5f4NIhwvGFZ+IXb74mMazvx0Ica3wfhuTxwWaQorLx6S9A7BUIJddGASs6QA+ExEEeOS6DOXYxo/TNwzgbWAdx01dmgLfvZhTGmePYRQpsK67Qk2uK2QD5tE4IWxuLQbtitBnV6t91rLHxZHqxceEqbQ9wxqXGtrl6cL7Rd+oFoaAhLvi8ugHyUPAaL/NElFgMOHviLYPLgpYFLda7MugMSPF507LKnawar04BKW0s/9qw1c1NR/kF+CaRgHk6uN666Aklg96AfxREG0H84X8QgBWY5Btty1Bq7hEgytNjBr8xvoBuUJ08rhjaMiy4OUf8x8/N7/LUtiGMeq3QZfx8Ec6Bj9gU1rVZv7WVYh8cu+OETkbdz8HWOLqkJCFhJJzMRRoqZ87OFWhoWsIURDL9pWt6ClN9htRR9cpRwGbOJKU/2yHQ9707YdLYteq4etdus1JVlS9cKMuqPmXB3hB2ERxkXgGEZtOuVqrrorZNfX7maW4CFX/iZUVB+Jktxi42D+a3frGmPNilENOypglRyCOMNqgIcWmTibqIOPlw8VStXLLNnx6lNHv7VIxStFKsZ+nMQoeeC3SQirFZSsLlR8v3xFqFu128RQ4CnKWBNDdIG1X7XmFj3bekZNj55569XUV3698mJLDCURbK+qM89nQKdefS0QGdX12XIdSWZws41xs1XLreFQHffHNoq7CR/I95G26/AnCRhjNx2m4vqOkr3hc9dWo+cNm0QbsxJruJlVxai1JCRLKKrDgTY9DsCSlZbveN68DVNjONRWQ+tzXG38130xDPx0fCvcMyUsAIosNQKiM34YkwQMlHxbMhKk2zM/H9/iCxTKuf3ZgrHaIS2UkGySDmTddsGg4qAat1Q1YXpvYXSZ+OXTcKAJ7GMfZPq6gIFtLTTm4oDftIgib27zDNlfj1tYVZzgtn5PcWKJHMAdri4H4Jqu297fi6AI+pPRanIEt/BsnlffBfhB7POQwtBqKLHKvShU1FS2DLaPIIBnRGGvNhA3ZOtId6qy+qumaTcvGwPETRQlkjRX/J0ppNbs/zuUsgJSA8VUnxXxBj9u3NF6sXFzLXHqTUlVmdWyhiU7zDTu66KMpmLdRB71Vp5BI43qvyMt1Nv970YQ/4egrTFlUGbq2KFBvF27cmpWgBxu3aoZayrN6tl7+SiY+kWUi6kfRSPfUJEqzVKaeCctkk6DmG/YQWsUnFvVZ7dKT3WRXqtVp+uyk3IkZeF/BqF3FrIYwPpzVLOUDIKr4+Ojsy+nJ2cHR4Nz7+j4HBpsMsZfxTBzyw59cnZ4cOJ9OTv/dfjx4HBgDicPZnPsOoiyQJZ/ezAcyG7WtVnXuzI2jmqZSJTyQeAPHkWXvRzDPC3GAAKAnGHZO9IV+PODL9YYqQcTUKV9Blo+H/xyfDo4WrGSsuVoFT+cfT4eDF31taaJ3ib3IRmTq7oXn1eol997WZDaVc8u3per3lY7yW8lFTo6uDhYbZrMMKDKh8HFAVX76/DstFZPtkdzC3Ifayluwis2mN8GsyAFWkLeJ/H5g7lYjBYNS2bhjFwzL+TmZVWa66q1cfGUK+Ti/GD43hteHLw7Pn23YgN56me3hkOFCzUB1tFECVl+1wpfs3E3lC8GHz7Krq3eYL3Pjz8r2JhrLL2GQIXTcFSgZ+fdx0/iUCq7fxCHt0V8l8GGo5maa9YCsjoi6bA6fP/p9NdhG8i1V0QJaBQatGQLq0Faq3148ml44d4sOnRL4Uirgk0Ozp19aq1ibfScBanZ4YcD+vPL8UmN5Fm1TVmsnHDzwOvIoQ/frN4yiVo/9lTMAi0TcrZkT8tGO4Vyh+RqL1If5RWJKt7Rp/ODi+OzU284OITeXr7e6e+UUswL8VrMQGJGK+0cWrgLFtMUWI/Ight0gItOAsLPDIjOyI/8eByIZCpY1MmTCGgSPvqDAHFTchGyohxvnylM9U4GyJUujj8Mzj5dyDG82pFjACWo7D4K/AxkHjl60ZmnwT2MIEOZCl74I5Dq2CufYSncUa+3Zmh5RBQFAd+Ps3EyIU2Fl8p7Pzg4v3g7OLiw+n/9Brp/IT746Z3gVYJpTSPgSSxjiVvQy/JR4Oek57x+Axv2w8G/eweHF0gL1Or89eztECFaAfOQ9Y+kAGiG8wBb3BMwka1RMZ2iqzwOvubit2S0mfGwM5ElRDUkskCBexhNFoICPYkCXurPQTpKAAQnyQ0Rkj+AuDkqbsRpkodTkDERXpm4gLdY42jw9tM77+Ts3XBPjJIEdQIULAKOakBXMghTg9ODtydA1cqyLCjsovT/HUowO4Row7/BRD9gAULWPYxEaNv72QLwcAbax02Fp0Bo0adOcRbZnilzqICK0mE+TuYLRXL/7dPgEwzxbVuHmilJbbKmavWtblTGsR5LX2ZP+NGDv8hEAeOOaIDD4ZEY/ttJmAcU24ChC7C8gH7o+zDmRDsAy2xBTZrU1A8jkLeyPwuSIh5CaBVbNqfpki21uZiTMyRKrZTdJMeWFHM0zNvSbBw84F9aVAozOU1i1SSHX+gRJnOAIew8+K48wcDK/JuA9JsM4eED8j5AyUkI+xCwGAiTKwKlr4d23EQJEBZhszwpy8AXXQqri4rVMxYBq9+lXGf2U66+ITL0RMnke8Jgn3VuC88q4mw07uATVWGN5EMX5rbqWcvWE/oCapqfXDHUz+uWEfnSVANR1VGN1LQcW/1QLXDxtVUAqvRcNUCv/BxVQK//DHVAr76+SkDQXV8twGrfoBpQry3CKb5/tpiJlb9F1KT66wuMerXnCY16C88XHAl25i5dg+9VK7M2D9N2uyPerZXSYwEUV9gkU1V6NnGtmlhKZXsaaV2HgPZcqy0dzK32aoDmzL8LgFxlnUlPkC3MS+72L9JCRdPhZ0UjHujYcQYsGijsOMgyjOjT4hm49avYBu43AvY/F6g9pwmpJwzjQfmzNAj0RE3FV4Wqwdl6ZflEH1OjMnUVq6U2l3mFJW5Z3mppQSgfU1zmeCGGwcyf3yYgjIEU/YVF/xMQ43Ip4aslVeNDZYXfnJx98YYfB7iq9TI7KPzvAGpMQo7D/XR6cvzh+GJwhC6JsYwKBS1pK5sHmlwke5XgrtprUDZInpNVgXvJqUD5MiC6X06w45oOgA0G4dEgPGdLKAPiZEgWFEWMMi6qmGMNiLgPMpwMaIQgCafh+C4K9DlBYZD/MliQlUZpTl/K4hfhLEiKnGz5QwLaFwwdmCQ34ryg4OJycoCb57aK9zNomFqRavUu3p8Phu/PTo68tx8R1Ls7O+IH+Pfl6z6tIv7+9e125q5s9LH7kjoxAM1jOR9cnMNmZ+UQ9YowD0FC/BF+gkwmHW2vQGjO0Subo302z5QOsjULZhiBcOFnd+Ii9aUSAUBHp+YWBsaLCCXwWQLtJinFwedQmAKtruV3D7UPA+igad2hCRtfe2OKRUZlbof7fcdCrFSH0XycJlGm4Ru/8fxxzsEauA2vYpbhVnrZOCBWUjIfofSPgGzeZP7GfyrtxA6CH8rymVAVOBgpTbJsax75OfyawURmwHXDURiF+YKQyQxn9adBbqokStbGzm1RGrq+DydBospSr+gu6GfFqJNuXmJM//YP/7p3tfEvf/mP680ejbVH5bruKldX2Y9UTpQF+2jZmSvvnewZ31zuAcpd4whhwgServgLoqFAbVCG1DA0RwskgB7bWidBFORBh5wJKIN4sDHJUVJBFy0FjjMGb6kZ+2DBBbYq3oaK3n5I0EsN4jDrxnTKwqgAqmNC7yNlwA5KmzZiB+gfY6ycZrnkU1A4RovUDEhMjGcFUtgU2EuYcyPRoi8ubsNMlHYifWx/L2BrIRZnxYw4Q/OxgjpUbP1KOtrqBeu6lvgFjVXLuJmDpTYJMEUcgnzoSayZKuv5I56V6eM/rzvd/m3w9XLvT9dP3qMaMlpcaA85Bv2kAIAfgiUgSJbbIqpjkNpgtCG+oPVHC5q56AkuOi84HQchGbmqxmd6+riqjlH2tJEZfX0ClhTfMY5x45Mwu6tK1J3CWkN1IZLRquPsTC4p0zKXuCF8kA11mQNocd6BRbokLOTNcw2MK0WHyR5IIkXExzF4ewpaTCJbj3UQPO2Jx+DJ8HzX5eD6dNvRtD51R/l68RooXOBwCtdWC3J/MJEaR4EfF3MZf4ux9h7QI39BLACEvvLxXnmo6xIKXEs5pesgWMOxH2f1U0Wi4/YQc7Q4dSpABcOdT0qgcYBpPAaaD2wTzWDqDBPq/gEsY3UgojokJcIsK4BUQWNkDLy5TWB7jZM58H1J3ugUFfAejbBBizNRZMjrHcTbTb8qCEGfLLupE2cO7VGZg/B7FXSptQEDXiEE00EaHWGYpu3JRNzWgMylkr3Z8gs0LIMkE2faUk6U0RmhoSwn+poyms1VTSNAHVgzSg1QuJNebfwfVs9/uLoC/Oj/8L+QS0OJ43enZ+eDw4PhoNtrrFrabLA2olVb/WstmgRHC3wtC9J79oaMginK0HLD7KmCx9PK7jRJAKK4Nrc+rKsyKlDMrW46wt6zcREQOx4h3a94NiKBa72QLnqT2jGIViNGM7LI1myqRNu/3kkZN9vQh+zHDorhxsxQIDzRAwJj9fIvYsdNGktOXU2xiUGrj2Rn6MCY0vpzJz0FPEeVil0c8rpei3Nam4m496NwUi0irY5GzYCxPsp2n7QjieVa4Yqvb0xcsmRYyblort60ZWvurGXtqLPm1ePX//T1w256FUifs4bG7JsXEotVvH51QaPsVQoZfBK1UhhpqxPhq6QJbodoCYVP7VS2w5aQTg7L8+tn5Roj22sBnATYsh0bl+rBmzgiKInjMQl2XQiCN306W9DhUbjOQ7iWCl0IeBKa2AYtiZ/m4RQUWYQXNvVUQ1n8OPWrcmb18graP+6LXUdroBLflQxWlTXR2zH8Q1mwmItHWelJccDRdiXHODQyPmqtHJ/nRcyTlwwGT3XOkkkBxeUpeYz4K2LCsFB6SV0OyyUSHXQH7G8jSm5ugrQxJQBoCu40AOjFNB60JwKwjv/3SiFSWpio8haKYaBJ3JQOzT8o68dV7MHzut3iXBkuYOFHxY0HmtYoCialux0pDHLnAT9HKXGi4kNxj/kZPoCa6G3pYagoMGIgBei/n6AeBS2nwRiwD/uHjYsTucR5oPRLc7m+lrYej22T77z3x8OLs/O/cXSFHHkp2rvl59LyEuQ8k44xE1O85smgOAH6Fn29l2EJVBUhSMEJPhlgyE/MGJS5HL4m5PgVhiKLEuIa5puFYfA4OjVWba9VUQ6XzgiHa4rN3MW45andKBlQKOyhVdtixOKUFjXqhWkuqv6gM9cs8bOiVwLXJszk0ritNOckC2eMcjAr95qAbiDHYKyFVMnsUVZIoXCo48AhEzvUK9BnMjwV7qPtJgO0Rbs3CZjEORD9EaVFJ+jf9JHKGFkQnJhiIPIyTDEKwwpwHZ6RhwZBP/eQgmS5P5vbyTUkPBSV6Zdf4uShQ1a5Kf7sXG18/37v+w973w+1/B0ekr7UQ5NTAv1j6E/HhNGXcA6kGI/JzYMYCYm4usIT62M0uDL5ApEc+BtGlUSCwpGIngJQ5a46Gwpl2TS3lYmqHtI0D1VVHCz0Aij/I5sYxQ/i5zf0gx6v7kehOWrt5sHXnM2HllWWlFoqQ1EhoNwlBRBgv8gTgL6MJkkDmMcYYVDEQTxOJkRRPsUhRnCBDOOnlFkgCm788UJIoLZNGXhGn7vqV+PrOt9PoyK77Zgz78iuBzH9m6YJ0NmzIX1ZdnBmDHgGwimIJmmn6gYlX5oZedR7bLBgDbvIp1t/0gkNDRBBi4NGVgN/+lQ5ADI3BskKx5HtY8gEwg1D5PuTQL23xI0aKMqmmwsaMFmTREXBfRB52WI2AqpMP1xG++geWSO97hfzeXU6Ba3Z+A5p8/HpL2dXGy6z+1XxcvfVz1fFNNiZKtAFkV6XKG9T5cmfXk3g38nOpFYZxBnYJF8Ozk85Bg6/HZ9CSw7rLo3jJ3+nbRyD8/Oz88ZJ/PH12F1t+OnwcDAcNlfceWPRbXq88/IlH22g/RnMQqLa2jL0RO7fyG8z9G3fBPIXUJExEEoUK0vRhcxzLrvaIZofFD/BbjBcgl8dzJGeZVJ3gC0fli4sDnvsaeQABS3iByGxGM4nRGOg13KnO6xdjTRfmfGr2VCbBAEDM0rTmCW2WRqtYV3CT472IQfzMO1ZCyj0aDZ1tVGWhmUVZZql6j0NEt/Rl9prWDmq6N/UXsmlxNfya702SspYHVkYMS5tyE8at9Bkzb5Pi9lxqHLoctKLovptS58WLI2W58m8s2OYKmBeHgeUolbzCD+frlk4g69KItPpJK9BTjIgV8mersVjnQB1n8SjbB2+SQA9VWeA8POC1KcU2MxNopgx8VxpayNUopPqwQOhLbmZNLC5GL6bB1fjZsYbWzYbFEdNed2Eo/vwom7nqJ0cMprrLjd/0PZK5rjGRlWkiHTqUfGy/ZJ/ddEqMW04Uzftt01dL7lyJgOmcTEGPS86bqJmUq734c3tFixjkqLvN9aipcUM1dEc7eUdGepLyx1grrMbP53A2wxD3VmOZtXMFE8rWqvYFlHacjwGfVVQl6J1McYoo5XmMOSyK46+eYgle1l5lKx2rDLGIxB+QlSiJqF/EycZCHm0YToKpgmKsKGCZqWNNA9Xkuy2wTKrUqN98NN4pcEqOxntZhOsKwNTCQsrQpKkt5UGR/LmNw1NSh8rjw0EV51KdyjeZw8d9hwbQ1Kcy/7gVEO5JaaW0scDNCKrThxOoBFTjG/k61LKQfeQwXcut3iM101zIFHXPRE7hkSNu6JSsyLKQ2IDGIQBeJJMHbNabQaVm8s5/poI4z5GX0p7pyw9xcapD2g4SScw8EWA9ryqKs5BBQSVRNnKjGWzGZMfB5eb8GTzWjFl9dvFmcselQBRtqAkIGzHYtXwmn5vXpv8Gp5LxIVKBr+oJF9kJGx+pm67uks5ZVQ2txYx+0yTTRGWt2FWO5LQbFzQpRnqR9kzb4MIFBoZ57S+WZP8Is4kqFbe0yKNonDUT4O/FwEGpJqPiczYD+cwaS2PajGapwkylBVNqA7jKadVvapGrKxh8icbd/W9eTMvvDCeJmxvMemIsUbHcTYPgEAIjhURt8CJHzB/KCLu6efjo+MDOiyFpcw9iM2bAvjVBnZLIVGwHLD+UI8DPq20V/epP/NmIyy1Y7wgG3bKtVU/Ty7bA4Yp7muw7ZMPxoPH8yLvmJvk8mojvg+BUW5ls5AVzq0tWM50sQXj3cfx9hhB+xQd2ONReDKeUtVggrU/zu57cXIb4Akb+FLEGMi6cW2pAkgP983QDSvGDD9zTAeBBGOu3rH/hbIMFnk/m0chHh3r4Tn8mmZAtUEl2BevbJspLM2lvhrXZJeD4pc7dhJPLlquCJZEpweX3r22XYdcXK1T1e7L6xXNWhpNwcY0o2sMPHjs+fcg2KCo0mB+pXR4GcLg9PPg9FDcBvfj7duXP71mcRkP9CH6ypS7JAnBhgMBSJKKZntWO07BrKfT2Ty4kfigesO1r5a7h8Y3oAr7WjNHg8+nn05OXDQVB8/z5sweOASyV+GErBdL4WuH+4wjjHd6T4SSt6HDyHBQgFCbzPD4JG7zcwofTVWcdoymwcoJlYXw05eZmZkCI28MYJHupP3gSJ5Ey8QHH8Z9H/oqdfEReyxhVOEsQ/9ZnoiXeqw2hXnKM5pyOzGQM2qmpEy07mzFSKUwgREg6A+cqEO0mZHxWB+Bw86BGvXh2enhp/PzwemFDKrHQGKlMJKhB+gpiHxeJwuiaU9ikxdO3H4emGcCQgKeVUxSDxhDezgVfrDdvqLbbN/UyLhWDqOf8A0iq17H3PF9abcE8sFknPEWzZjVm4sgi/zqhSJChIKnAHld3qC+ymlD59V3KD9VR4K8x3KAKiXOU0PsZK11RDct7pJRV3QezWluqlmCqNKttcHHitwgt9ZEzbef8hdOeELuYLMgp3V2V2bRjH17xkDktjFclFYRjt32sijJsz3m0kB7K75vKwA4h8edPYlhu/zlqaXR5gBwo8qUdEmPt54HE40wBF2OiCSRaZT4OXVvd8dhnBhOb2UFJ0lRGelZYEE8s5M9EPJZyVu6PaKNpYRnCKS6DcYxgrrdpdrD6pisVyEJNUGcxkAWbV/qlAI/g6+3fpEhjMnzhtnEZCpdongammxmQjvkJkaBP8a4EvIraR3s9pUn03WIjlGPXjdmNKjawtTSzQnR+9t6bshtbXAc9wOsnBIQ9FsKVn1hblriBGZ4fzFSSeobAkC3f1ildUyAqXLl11pqSGm9WsuYBI4Zgsz1ZrdupGFdqclfUNfP9NPoFdRRm2b2SdFJbFwApiSD6Dnr5WR7FmYZJvin4BGV4bvfjIZl8leyGaqIBUnoLq9Ns+tuXyaqkMnC+QS5qaXiPpW5oF0otlFtXudrO1HgXHPRqd2PP6RbzhIrp2LuMEJac1SK7rxrzg6QnlMapEHk02kYa3oN7TTlgiLKtWryue63dtPvc7srd6dPHfbgrzHQaitLLF9OQdaKMDYu1DC0iDp1VrFF2LfhmFKfKsrICuGt8LQpmrm3QmEzx+IqNZZkt1cfS+2hc6UcxB3rc3IgYUN4ZlNye0KEb8o5rwVocy9mvlxHllx9VitkiFQfZyrACeffb8oQqUFljXR/JWDadwhnqsSGVt8Q9Q7XSuqob6ajYGtSgMqN50lVLDjZI1PMZqeJPUFMsneuC1LyoI6Zgt0yBI7nRtpuSbnN4cV4hK6eKGJsJ5sltISiMvErDsoxUXzc9yeTDpZ1wKo2arUksrw5fsk8oLtavVqwAkoSlkSoPGXczNKg5KoN5yUSjTtJ2z9aE3woVLJgI3FMw4Z6dm7Qpgs11ptF5R3k2pS94xleQfzQUYx9Mnr28WBpZ9qws1HX2afiMmlRJSi3XbagVCSQPjrwvWsmOr3N87kZ2mF/KOTK487LNgylrLnuC3FWphAKb2J0I+++/GN/B/7bFdt8Xo0OA/kZnfuBtVeJnwB7cfeAajS+1eV2ddqxlfqtwjsJKDCDckBszKmmS4p2OUTr7VJi2shlKthIXyaGSRtZgnhdpUrynRgu4vFtmsThP4IyqtKR2sT14UMXzn2+7LfcFi4argG67MCVSlu+Wwaq9iTfGkBx5u7UTWVX7e2skepafZpTXuNHXQ2gsOJ3IlSyWRWTjI9QQ8YdUcTjwMsTDiiX5i7Y0HmRqWRkVxuUKc3Y09LVy1FH4yQCEgxj2ZqmgdpwSgGmWCSHeU5XI/u2uiNJWC3cpzRCodPANFHVonNK81JZtnxSK6t7MlqsbLV6mmvDrlbZ2Gu1GLxUib7Vo5Zgv3plijwrxKhWOg1uMA4YgOrVCldlNUuOulmqJtHXqZwrm8hGz5QhFQlxZRwykwfZY2+yF63Ra0t+SavvqmtLTkPZAUmxBMvaoUG50w5l13igkB9b/ua6ujnVQOon95Euaq+UGLhlmu0zJAYSEybFbN7BTQdj6WGWFZCj9l8+P44Iv9fPHzNFYBtcjZacl6d0S+/NZmYQkfoZG3iDDlyk3TUS8j9I/s1I/nujbF1U5h6a+Hl1hF6W+2Z89FA69aR7XzI70H8o9aKMHcJdsNdok1eeHMOE7A7cqVB7SNG7fizeX1x8FB/PhhcoCr4bXAg5Et7MozR5ACVUsLeZMz7eJyFmn02KyTRCb9gIUxvItJc1lJe2C9sxYoLFEAXUQxbGYU0frcpPjwo8hv+mji8wE09ybbNp/KgZ1Tg6w+gTzHrr4IayqWMswIfkH2EU+duYRLbzJcQMOZk4vRC7IFX/WcCDn17/WXz96XUXI6Oj4Esw+jXMt9+8+mP/1U+i8+v7iw8nPRGFd4F4F4zvkq44BGl3FmxTOiH8Twz9qZ+GskqD5elgjHjFIwI1OZIRQduM3s4q/77FntSt46O6gGJWeDJ/YuweQi/kE+ZaotAGIJdkOyO63VWnGSqiX68ulwFNvmxU27pYzAN1bKs+x9oKQzkzQqZ/LrcSPOats6/G2FPd7cu/1oAI5a3W4CdxNPjdIykmKfL9Vztd1uYyl9GYL0rAiAqUpDAc/eWO68wyfkbJZEGZgaA8esg65RmPNqjx3GnTlCp11sG26qc59EChPu52jje0DhfLcSMV98OZh+miQBmUpp2SHNVHIk91UiTm1cb7wcnHAbEMdJ4iZXnE5AiT4An5YrVv8ZgtThfGXj9p66QGrsl8Oj95/lzkvSa4ohMKbihLoP4fdJXreL05AxLHoFWU+UB+p2nLvGg021Vo57Kj5CWEvmH6HCpszb/qsTbxFWfcoBYqhcLDPSq9p5XnVIvRwY+S3riKEt0qjY/2eRkdQnh68PGYMIKFuLck4tWYGQzCm/sL3G3/bTRBpzYml1iG3VtrXB6sGge7QgEZrdQVWHVTVqfGv7v9ft+K8MxUKIkpDV1tbPvzsHR9KiTgdCrlWphHVlI6b0n3l0sTotJtu3wyqNJNjXNX+AHRLLzh/AhVdfVQwrlnLZeF1BJ0Ktzfgp1hDasG0gA62LWbj6r7p83vDKBpucCtvmuE6WqDr6FgVOduZcZ9eSr1zyJbxGPMXwEbwrr7mUwjI7UjDJM8h3vULTYE8P3SSFPb3RyJUu3uDEZRGRekEDzDpInjbDXB10kBWMiltInzNLlJ8UwFokaV6R/kWYR6badXu1yOAqVjPfJEFrg0dv21wmKHgLUqjmsmlp6oI3g7xGV5C+97lbnM3Cru4nKbVOdY3VjmpP8Vbfj5lXivprLHt6KIF496j5sU7QNA+rrZfQIyYbybj/PNngC5uLvX350+fd8V/1uYJabzDEtAVfjmQjJKGLCUGiR3Gk6zfUAJDY7InGVq1cckimTgIWq4dPNDGU8qL63gDY0xgjW0K4UTUnwMvKiG9a8ldu3beu4fSqq0/6jHgvcxNSJbUjXC9WTytgb8VGNaRmVv/UyKWrbQ8BtH61VFf0tGtF9w+cyiY0QHo2yJJFh8a9cq/0JeTo3HBysjjTy8ASSW48r4XfDVBxJI7VmdynAzr4jzMFJQcIek0ag6OKMeD7bLWFqTrTSDp/gXqwunmxQD9gK6D4WRT+/DpXY5ZCdtdy5l3W9AhfdJ+lN7U8Fc25e4QOJHsfu0Xb2lCHUJFty4yVT8NRnp9Xl5aVNvavXCPAqgxU0oDe82u03yYBos9xKocNtqn/YEd0unf3qinIP8DSIuj1mmRuHD0HjgEfNLgSi0xxGGmBmpv9PCUooRJo6WW1kOAxkpAnwcrMBQvklsVDtnT862VkDbLnsVEGrFTHBgWfNJrYIOKixu/HaJkKvyO2Ml3TxvNYYTvJZ34wBK1yQpQNKxRPMSKE/68nUYhxmqT11bPl2Dofhko5E3XOgKjLa3lyJsMwLKOWbG9ADrZPvCZwaG2btpwnQxJQb7yfQj0ijBicJy6xjUfz2mfhMu6TB+Diq93PVltgnQa6ipijqWIyR6WFE9iTE2nL8BgeQsbAQqc3LzSChFj9NazOZ0j4OSS2rXjFFHKoe7SP0HiTWmh5S8HLU9VerO0Hrgz5DpSZUBVAm0GlageL5NWEutjJ9Go7AK1tjcBt7iNhAvNxYq6+Cjbf8lyw7IcO8+fuIjAdu25HW18dSasndl0+LrN2uYFr9rMy26IWgMSIvu0fCF/Dcj6aNDvtDkp7PSJyaj3xC758xt7QgfORxnYJLWuTv/o6PYX1QKQ/ysYvBq2/uEx4z6EpvVduOD3KCekOWi1ZBVnnvCZ3QGnOzfvGHlXkXStGcdh+zhxQq5YgUogTRv1ur34Gswpjv01LGkLeQ8eD0fnkLio2nl/XiUV6m80wwXrGSJGmHOJI8ABA9B9xtXinLW4+TsnM9Y4GWg2S1005/d3RuRFPoJJ9eo8RpS9rwytPcJIpd8P6kukWiGKcQmZx1birnWN/vUG09vZFlJaNUtpp48vQpr/6hvW6blVfuKh2ntVmygLKUxMiPonZ/Lw0S1HO0WIAwehUiOtNi+Qd111gSQN4zprM2jxKIn77Hq2kRWrszHDFdqGiG8pO2qdZQpppRaqZxCr+rS1godSczXjrJU2cx/zxjc3b74JcjHt8QN5WG+PXGRLky7GN1bIDpERkmQ2qYoTuOQBeiG5d3gsDcmmhGOqYwOuyKeVJcR7NtkW56PwDtDEMKwr2+CzuvlMLXRzJ1d1y7kzrFbG6N5oEx9tLymFVTpfIo8rWMdT2G4wd4E3nGLeTOBlKkjK/I8UFnYbNaNG/YxMBXs17zjmsE3b4vt08N8HYF5K8cCGFNCs0EWBcG8s9s3MyfB6MwVWClgopbi2IRCryI3S4bvYqn4afGGXQ6BaoiSdlzLrUPsR1pmOsBPu3+mbUJJ8OVW0faI7SxQn6YdU5MzrYLuez4qKGABe4Nob0FEt/fGEpPLy1fTV01qhNLKKsG5j3e1w7Srg3NLBXAXhLifYO6yJOKrbaW4bGsKse1enwDhvlOal0v/UL00Y5FcA2rpG5FHz8InhkmBBKWm1DNYsctFkOMtDkGWhZTDNlYBWikDTap4OtZx6vUGfGsxCl52pI5biQdd9AXoNsAfBV8c5WzWNPzV2mo0/zlk+6k309PpOzB4W3Tw5it5AZbbfdWIy3/cedOEy51H7JvM9eLD2y4ZYQHg7FOq874Kqb+z5Ag6T/e2CKOJ+OUXTHigybHjZDYDFqbJrDAvuj6KL7aScp/SrLWX7AeJi8g8SDBOp7Vq8MwDLo41XukRh/cejmFcK095A/gd9/K+//KnNwDo94PPh2am/cyjTBGo58NfjrdXzVIs/gHW0J5WddHATgkZMEGGI1mFRfpkV3UJoWzHsR/vPVC0OEBxY2u8d8/T0XNBUOoJOpqU84/5a/lw/HcujwEEAMGudeDN4a9s7hHU5K8EH245neoN14ZwH6SLqY/HF7Q+Hf09b/Jahoz/zMm/ec7kNbyeTbhdI2cIJmoD9iF/LQyA4fXytEk/nx98qJJdUDgSUkncxqefjwaHe9Ut2JnAFDtlkVGS51EQB2PzbqYmoMMYgf/meNoLAXD7QBqsPHFXTHwYXteYkV6aEumUZE0viFlGdapAd5vzxm8fwD1DWqvb2L/RzA+8jjUT55Zfj8rDpxhB6cXBDR0V9vJMHSu8C7x/BGniqqP81xLtwnmwt1srV+pa1eNrgyxMitTLiICRs6Oj6cbyFbb/8jW6QOrnfE/8Iga9SNJkmVyGDUnkY6/c6vKSwprdtZGt/OTvgtyDtzDEOrVPpk3MBvUh0wDb2XwspyOdTehlokCk5vQdomOKGbXYFJymmZTnI1nNABt6Ms+1nmrn4/HHQUsKHiNRz6iYIm/e3zVAzQctRpR01pQhNFQs0tSbj6W3yno+nWfO53wL6T6lkroysqLWNQfK2Y4J8YAR4RRkymiXBIel9ulPPb2U+mC+KC6gncqDce43n8prPzClTZQxmZuX2aqw2cvdazWelgNT0oRDEQSUn0W2666xxtEq95Eqzv5cgwStTAssjOVrm2kDKHmS1IA2Tf696hjREoRo6BXPXzUMhqXcXq45tKwSD5foFjSxLXa9nZ0d/N8lUBuVeI8Ar+r8/HN/pydm/tfODn7plG1u08NdfChpYLdLsrEkgk3tS6hCDxpM8dfvhGxajRVXDj/fjKL4iZOHJsqjfwBPOlh0y6BZlA8Ow+Lbzv0jPbaCvhxh+PpnqUexXmElX3i9Gq3oXrmeS8vTeu6Vq7m0vFzNPW0tl9cpj+fpgVnNtZ5aEHeJDxRZcU33bjJfSKlkTzwqcHG4FkZrKYDs9XemFKQl/oMvdValaerum5bKsZrMELCthWCx+1SNaR/QqybwqY9GHHabyYi9xR1FiTk++GHedo+Cy3RGFe/CKLJ3l1wezh+8zCZSdQG0C2WnhoVSOOP2X/0TTBvrmzVcTjT8YJwiworfk1j4HV6+3mbHQ3dDsx2P3jbb8ZyQ51scmsBbSazSrLmm0CkuYfiYVHUSwApZ08VUvP8/LZjKsa9GREciOe5ZM9u0hNjhpQGO3su8TXpXrzGv1Hyh5V9kk6Duk9T9NlXdtkz6lsexJZW+bV4vMaxnuy1NV5kMnXJY6uxqzfa69lDzdmOdYVEjyKH9TsXClTa8dZUnMfTR4lc5gm3L3ps+R2ktauZsCzfMCL8aNtTA2ythavX4U5+vzNP83+hGS1I/Xei3w9KyrOXBdAu7K/o1JQo2eDDXPEeKn4bQepmgmry4XpTA1pL5FLSQA0cWBY6j5ayN7HwUaCqHbUS5mHHZOyA3i5f0HWN9UnzTrQf7TCJYwxAz5u7rERsPtxhiZOSpdHoO2qOEaBkql9krhyhRJSmxVg/wSvk2tJhvRyQvhmw41rlhYuUE0S7izofZFE5jF76Ui3Qtgw0avActBzAqWdHl0ZDXTEggGNEq2F2JJPvyb9OR/99vlnqc77d1gzZHP154XB6ahh+dTD/MyldHc+oqq+H+vR8VeHWl25bBFy7r7bfOdYXzMY3dyDVqUrc11K8FoeOnwe5s1d3pv7Hpj7uehvPuO03R76DtCyiDYpj26HsQPVyyl/q0KSc16fcY2t0TX0D6VnffVmct9FzOTS49i3JURBMoUkNKCOJa8pRBQBFYIYVWJXPQVdA8iiMxkkojvZQVGLdsCtlsGyXbCPD9n/5U5/pLAvyDP+2o3KjQv8xjjAmV2cxKSph5HMSOFGrpBd5LBnF8pNrRQyHXaAdGt9cuYtgv5dFFfPfhLfkqumv1yJFx75V740AFzSGXFY+bfwuyTek+q6eDp0zLm6eJ6KDb4xcZ29PdXG/Ohwo5xgs1d2cC8CeJvsB58YYSPw6SIosWiteu1ecR3cJzktzIIHmZnBNnzJemHonOZ75HtEvzlw1qV5Hy5I+Oh7L4B3UhUUZ3/awOBQOtdVGNUNPcPfrOKSdeGa5JPnElOiQJpQyUaoZxTaYzclXzbcEy+nSfA9Bteaon0Pez38EHvS5mHACRLnal3MnZomqLGHISKu9hbjty2yWg5eRgDP9Od5EcVKEjA3XQjJPy4jU9ChW1g7puvdN5FNsYNNpDniXzUfSdSuAv6IgCR4XVYl8+nZ88U1y0SpThLY7M3G7uv7yL9dblqEqKR9PU1mmVNVlpbezCLXKBtnhuHi/BoC+y5PPao+/FT21sXoOQ47SyjJnSGXsNAWQWJkq3VEvehrpJlSW7if0vXyr9fNHrn8VFON8Tx1O61nZLIcwCRPM7TBIFwgQeOYF+6U5fF8qispXQ200rmf8mhjoCy0N/NdQb5sFcvGwetia52KKb+jTpPS/Ex1p4N9/Z6+OVwyQtY5Aou7oFGYT4vBBe7y3pMnAhpVowbTb7WE9cl2VQ5SBSzjem5+2SOWlmujhvXyszNdp1hrNiSC32sqQgftb0WOjeCj+3jr3a2bjMiobXoqyrPW2vLr0XZUX6zWdUW+tJL0ZZj36vUK/0ZpQ15ZOedDQvqb6GY8N2ajQQMfdSGT2x3mW3p8fecuit1AU8eRPEzKf8Z9a1KiojJSe6Q0lR3jWLd4YZl29Y9/5Rnmxmw4iLUkhnqb261EbTFpTBL8ilUFbrTs5ISk37xt079sj3rd+yrmnVkncWoUIk30vT1K/BYpSAEH0cA8lKi7keG7ASQQ0mu2rKUpkIVVOYb2IhigzaEMPbIicuQKZhi5I7zlnxeFt07mQ6RWdRjbTKms70gC2+nVraabbK4e2upMN4HrulPMIezytdUjZeQR8bT/8XkfsWVA=="
try:
    raw_json = zlib.decompress(base64.b64decode(bundle_b64.encode('ascii'))).decode('utf-8')
    files_dict = json.loads(raw_json)
    for fname, content in files_dict.items():
        fpath = os.path.join(workspace_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"✅ Deployed {len(files_dict)} helper modules into {workspace_dir}")
except Exception as e:
    print(f"⚠️ Notice deploying bundle: {e}")

print("🎉 Setup complete! Helper workspace ready.")


In [ ]:
#@title 🚀 Step 2: Start Autonomous GPU Worker Loop
#@markdown Launches the helper node. It auto-discovers the Main coordinator and begins encoding video chunks.
coordinator_url = "" #@param {type:"string"}
#@markdown *Optional*: If Google Drive auto-discovery takes time, paste the Cloudflare or Gradio link from the Main coordinator here (e.g. `https://xyz.trycloudflare.com`). Leave blank for auto-discovery.
enable_debug_logs = True #@param {type:"boolean"}

import time
import logger
logger.set_debug(enable_debug_logs)
import helper_worker

print("=" * 72)
print("🟢 Launching Autonomous GPU Worker Node...")
print(f"   Coordinator URL: {coordinator_url if coordinator_url else 'Auto-Discover (via Google Drive master.json)'}")
print(f"   Debug Log Notifications: {'ENABLED (Verbose)' if enable_debug_logs else 'DISABLED (Quiet)'}")
print("   Keep this cell running to provide GPU compute.")
print("=" * 72 + "\n")

try:
    helper_worker.run_helper_main(coordinator_url=coordinator_url, enable_debug_logs=enable_debug_logs)
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user.")
